In [3]:
suppressPackageStartupMessages({
library(gganatogram)
library(tidyverse)
library(Seurat)
library(ARBOL)
library(stringr)
    })

Loading required package: ggpolypath

Loading required package: ggplot2

Warning message:
“package ‘purrr’ was built under R version 4.4.1”
Warning message:
“package ‘lubridate’ was built under R version 4.4.1”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.4     ✔ tibble    3.2.1
✔ purrr     1.0.4     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“package ‘Seurat’ was built under R version 4.4.1”
Loading required package: SeuratObject

Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.4.1”
‘SeuratObject’ was built with package ‘Matrix’ 1.7.0 but the current
version is 1.7.3;

## Load metadata and celltype information

In [6]:
meta <- read_csv('/Users/kylekimler/Projects/GCA/meta_datasets/gca_concatenated_metadata_343_March19_2025.csv')
meta <- meta %>% rename(CellID=`...1`)

# Add celltypes from celltype metadata
ct_df <- read_csv('/Users/kylekimler/git2/hca-gut-atlas-extension/src/harmonized_metadata/concatenated_celltype_metadata.csv')

# add worksheet per dataset to metadata
da_df <- read_csv('/Users/kylekimler/git2/hca-gut-atlas-extension/src/harmonized_metadata/concatenated_dataset_metadata.csv')

da_df <- da_df %>% 
        select(dataset_id, worksheet) %>% mutate(worksheet = str_replace_all(worksheet, '_HCA_tier', ''))

meta <- meta %>% left_join(da_df)

ct_df <- ct_df %>%
         select(author_cell_type, closest_GCA_celltype, worksheet) %>% distinct


New names:
• `` -> `...1`
Rows: 3381939 Columns: 55
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (52): ...1, dataset_id, donor_id, sample_id, author_cell_type, tissue_on...
dbl  (2): sample_collection_year, age_at_biopsy
lgl  (1): is_primary_data

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1793 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): author_cell_type, closest_GCA_celltype, cell_type_ontology_term_id,...
lgl (1): column_4

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 35 Columns: 22
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (21): dataset_id, consortia, study_pi, contact_emai

CellID,dataset_id,donor_id,sample_id,author_cell_type,tissue_ontology_term,tissue_ontology_term_id,tissue_free_text,sample_source,sample_collection_method,⋯,ambient_count_correction,doublet_detection,sequenced_fragment,gene_annotation_version,comments,description,publication_doi,protocol_url,worksheet,closest_GCA_celltype
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Dominguez2022_Pan_T7935490_AAACCTGCAAATTGCC,Dominguez2022,A29_390C,Pan_T7935490,Trm_Th1/Th17,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AAACGGGCATCTGGTA,Dominguez2022,A29_390C,Pan_T7935490,Trm_gut_CD8,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AAACGGGTCTTGCATT,Dominguez2022,A29_390C,Pan_T7935490,Trm_gut_CD8,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AAAGCAATCATCGCTC,Dominguez2022,A29_390C,Pan_T7935490,Trm_gut_CD8,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AAAGTAGCAGTCACTA,Dominguez2022,A29_390C,Pan_T7935490,Trm_Tgd,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AACACGTTCCGTTGCT,Dominguez2022,A29_390C,Pan_T7935490,Trm_Tgd,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AACCATGTCTCGTTTA,Dominguez2022,A29_390C,Pan_T7935490,Erythrophagocytic macrophages,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AACTCAGAGGCTATCT,Dominguez2022,A29_390C,Pan_T7935490,Trm_Th1/Th17,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA
Dominguez2022_Pan_T7935490_AACTCCCCAGGCGATA,Dominguez2022,A29_390C,Pan_T7935490,Trm_Tgd,ileum,UBERON:0002116,NA,organ_donor,surgical_resection,⋯,none,scrublet,5 prime tag,v98,NA,NA,10.1126/science.abl5197,NA,Dominguez2022,NA


In [8]:
meta <- meta %>% left_join(da_df)

ct_df <- ct_df %>%
         select(author_cell_type, closest_GCA_celltype, worksheet) %>% distinct

ct_df$worksheet <- ct_df$worksheet %>% str_replace_all('_HCA_tier', '')

# Join per dataset, because different authors meant different things when they annotated "proliferating"
meta <- meta %>% left_join(ct_df %>% distinct)



Joining with `by = join_by(dataset_id, worksheet)`
Joining with `by = join_by(author_cell_type, worksheet)`


In [9]:

# Add taxonomic information to harmonized author cell types
gca_tax = read_csv('../data/gca_celltype_taxonomy.csv')

gca_tax <- gca_tax %>% filter(!is.na(closest_GCA_celltype))

# Check for any unmatched cell types
setdiff(unique(meta$closest_GCA_celltype), gca_tax$closest_GCA_celltype)

# Update dictionary for cell type harmonization
update_dict <- c('T Cycling ' = 'T Cycling',
                 'Cycling T Cells' = 'T Cycling',
                 'CD4+ T-Cells' = 'CD4 T',
                 'CD8 GammaDelta' = 'T GammaDelta',
                 'CD8 MAIT' = 'T MAIT',
                 'CD8 NK T' = 'NKT',
                 'CD8+ T-Cells' = 'CD8 T',
                 'DC Monocyte derived dendritic cell' = 'DC Monocyte derived dendritic cell (MO DC)',
                 'Germinal Center B Cells (GC)' = 'GC B Germinal Center',
                 'Treg' = 'CD4 Treg',
                 'Dendritic Cells' = 'Dendritic Cells (DC)',
                 'Effector/Memory T Cells' = 'CD8 Effector/Memory',
                 'Enterochromaffin Cells' = 'EEC Enterochromaffin Cells',
                 'Germinal Center B Cells' = 'GC B Germinal Center',
                 'IEL' = 'CD8 IEL',
                 'ILC NK Cells' = 'NK Cells',
                 'Interstitial Cells of Cajal (ICC)' = 'Fibroblasts Interstitial Cells of Cajal (ICC)',
                 'MAIT' = 'T MAIT',
                 'Mast Cells' = 'Granulocytes Mast Cells',
                 'Memory B Cells' = 'B Memory',
                 'Myofibroblasts' = 'Fibroblasts Myofibroblasts',
                 'Naive B-Cells' = 'B Naive',
                 'TRM' = 'CD8 TRM',
                 'Tfh' = 'CD4 Tfh',
                 'Th1' = 'CD4 Th1',
                 'Th17' = 'CD4 Th17',
                 'th1' = 'CD4 Th1',
                 'Secretory Goblet cells Mature ' = 'Secretory Goblet cells Mature',
                 'DC pDC' = 'pDC')

# Update the cell types in the metadata
meta$closest_GCA_celltype <- replace(meta$closest_GCA_celltype, 
                                     meta$closest_GCA_celltype %in% names(update_dict), 
                                     update_dict[meta$closest_GCA_celltype])


Rows: 151 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (8): Lineage, Differentia, Habitus, Status, Whim, closest_GCA_celltype, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] "CD8 GammaDelta"                     NA                                  
 [3] "Mast Cells"                         "ILC NK Cells"                      
 [5] "CD8 MAIT"                           "unknown"                           
 [7] "Goblet Cells"                       "Tuft Cells"                        
 [9] "Paneth Cells"                       "Dendritic Cells"                   
[11] "Interstitial Cells of Cajal (ICC)"  "Enterochromaffin Cells"            
[13] "DC pDC"                             "DC Monocyte derived dendritic cell"
[15] "CD8 NK T"                           "Myofibroblasts"                    
[17] "CD8+ T-Cells"                       "Th1"                               
[19] "IEL"                                "TRM"                               
[21] "Treg"                               "th1"                               
[23] "MAIT"                               "Th17"                              
[25] "Tfh"                                "CD4+ T-Cells"                      
[27] "Cycling T Cells"                    "Naive B-Cells"                     
[29] "Memory B Cells"                     "Effector/Memory T Cells"           
[31] "Germinal Center B Cells (GC)"

Warning message in x[list] <- values:
“number of items to replace is not a multiple of replacement length”


In [10]:
# Merge the updated metadata with the taxonomy information
meta <- meta %>%
  left_join(gca_tax, by = 'closest_GCA_celltype')



In [19]:
meta %>% colnames

[1] "CellID"                               
 [2] "dataset_id"                           
 [3] "donor_id"                             
 [4] "sample_id"                            
 [5] "author_cell_type"                     
 [6] "tissue_ontology_term"                 
 [7] "tissue_ontology_term_id"              
 [8] "tissue_free_text"                     
 [9] "sample_source"                        
[10] "sample_collection_method"             
[11] "tissue_type"                          
[12] "sampled_site_condition"               
[13] "sample_preservation_method"           
[14] "suspension_type"                      
[15] "is_primary_data"                      
[16] "age_range"                            
[17] "development_stage_ontology_term_id"   
[18] "radial_tissue_term"                   
[19] "dissociation_protocol"                
[20] "cell_enrichment"                      
[21] "sample_collection_year"               
[22] "library_id"                           
[23] "library_id_repository"                
[24] "library_preparation_batch"            
[25] "library_sequencing_run"               
[26] "sample_collection_site"               
[27] "sample_collection_relative_time_point"
[28] "cell_number_loaded"                   
[29] "cell_viability_percentage"            
[30] "institute"                            
[31] "author_batch_notes"                   
[32] "organism_ontology_term_id"            
[33] "manner_of_death"                      
[34] "sex_ontology_term"                    
[35] "sex_ontology_term_id"                 
[36] "age_at_biopsy"                        
[37] "consortia"                            
[38] "study_pi"                             
[39] "contact_email"                        
[40] "batch_condition"                      
[41] "default_embedding"                    
[42] "sequencing_platform"                  
[43] "assay_ontology_term"                  
[44] "assay_ontology_term_id"               
[45] "reference_genome"                     
[46] "alignment_software"                   
[47] "intron_inclusion"                     
[48] "ambient_count_correction"             
[49] "doublet_detection"                    
[50] "sequenced_fragment"                   
[51] "gene_annotation_version"              
[52] "comments"                             
[53] "description"                          
[54] "publication_doi"                      
[55] "protocol_url"                         
[56] "worksheet"                            
[57] "closest_GCA_celltype"                 
[58] "Lineage"                              
[59] "Differentia"                          
[60] "Habitus"                              
[61] "Status"                               
[62] "Whim"                                 
[63] "markers"                              
[64] "neg_markers"

In [12]:
gca_tax %>% colnames

[1] "Lineage"              "Differentia"          "Habitus"             
[4] "Status"               "Whim"                 "closest_GCA_celltype"
[7] "markers"              "neg_markers"

## Set up dummy Seurat data to use ARBOL

In [21]:
# Create dummy Seurat data
num_cells <- nrow(meta)
num_features <- 5000

dummy_counts <- Matrix(0, nrow = num_features, ncol = num_cells, sparse = TRUE)

# Create a Seurat object with the real metadata for ARBOL
se <- CreateSeuratObject(counts = dummy_counts, meta.data = meta)

Warning message:
“Setting row names on a tibble is deprecated.”


In [22]:
# Set up required columns for ARBOL
clean<-function(x){gsub("[^[:alnum:]]","",x)}
se@meta.data<-se@meta.data%>%
  mutate(across(c(Lineage,Differentia,Habitus,Status,Whim),clean))%>%
  unite(tierNident,c(Lineage,Differentia,Habitus,Status,Whim),sep=".", remove = FALSE)

se@meta.data$sample <- se@meta.data$sample_id
se@meta.data$CellID %>% head
# row.names(se@meta.data) <- se@meta.data$CellID

[1] "Dominguez2022_Pan_T7935490_AAACCTGCAAATTGCC"
[2] "Dominguez2022_Pan_T7935490_AAACGGGCATCTGGTA"
[3] "Dominguez2022_Pan_T7935490_AAACGGGTCTTGCATT"
[4] "Dominguez2022_Pan_T7935490_AAAGCAATCATCGCTC"
[5] "Dominguez2022_Pan_T7935490_AAAGTAGCAGTCACTA"
[6] "Dominguez2022_Pan_T7935490_AACACGTTCCGTTGCT"

In [15]:
se@meta.data$CellID %>% head

[1] "Dominguez2022_Pan_T7935490_AAACCTGCAAATTGCC"
[2] "Dominguez2022_Pan_T7935490_AAACGGGCATCTGGTA"
[3] "Dominguez2022_Pan_T7935490_AAACGGGTCTTGCATT"
[4] "Dominguez2022_Pan_T7935490_AAAGCAATCATCGCTC"
[5] "Dominguez2022_Pan_T7935490_AAAGTAGCAGTCACTA"
[6] "Dominguez2022_Pan_T7935490_AACACGTTCCGTTGCT"

In [16]:
se@meta.data$tierNident %>% head

[1] "Immune.Lymphoid.TCells.CD4TCells.Th1"                   
[2] "Immune.Lymphoid.TCells.CD8TCells.TRMresidentmemoryTcell"
[3] "Immune.Lymphoid.TCells.CD8TCells.TRMresidentmemoryTcell"
[4] "Immune.Lymphoid.TCells.CD8TCells.TRMresidentmemoryTcell"
[5] "NA.NA.NA.NA.NA"                                         
[6] "NA.NA.NA.NA.NA"

In [32]:
rownames(se@meta.data) <- se@meta.data$CellID

In [37]:
se[['RNA']]$counts

  [[ suppressing 32 column names ‘Cell_1’, ‘Cell_2’, ‘Cell_3’ ... ]]

  [[ suppressing 32 column names ‘Cell_1’, ‘Cell_2’, ‘Cell_3’ ... ]]

  [[ suppressing 32 column names ‘Cell_1’, ‘Cell_2’, ‘Cell_3’ ... ]]



5000 x 3381939 sparse Matrix of class "dgCMatrix"
                                                                                  
Feature1    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature2    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature3    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature4    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature5    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature6    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature7    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature8    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature9    . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature10   . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ......
Feature11   . . . . . . . . . . . . .

In [40]:
%%time
# Random subsample to 500k cells to prevent memory overload
num_cells<-500000
cells_to_keep<-sample(rownames(se@meta.data),num_cells)
meta_sub<-se@meta.data[cells_to_keep,]
num_genes<-nrow(se[["RNA"]])
dummy_counts<-matrix(0,nrow=num_genes,ncol=num_cells)
rownames(dummy_counts)<-rownames(se[["RNA"]])
colnames(dummy_counts)<-cells_to_keep

se2<-CreateSeuratObject(counts=dummy_counts,meta.data=meta_sub)

Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


In [ ]:
se2 <- subclusteringTree(se2, categories = c('sample_id', 'dataset_id', 'donor_id',
                                                 'tissue_ontology_term', 'sex_ontology_term', 'alignment_software', 
                                                 'Lineage', 'Differentia', 'Habitus', 'Status', 'Whim',
                                                'sample_preservation_method', 'sampled_site_condition', 'sample_collection_method',
                                                'age_range', 'radial_tissue_term'), 
                                 counts = c('dataset_id', 'donor_id', 'sample_id', 'tissue_ontology_term'), 
                                 diversities = c('sample_id','dataset_id', 'sampled_site_condition', 'sample_collection_method', 'sample_preservation_method'), 
                                 diversity_metric = 'simpson')




Warning message:
“Expected 10 pieces. Additional pieces discarded in 500000 rows [1, 2, 3, 4, 5,
6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”


In [ ]:
dir.create('~/Projects/GCA/ARBOL/')
saveRDS(se2@misc$cluster_ggraph, '~/Projects/GCA/ARBOL/taxonomy_rose_ggraph.rds')

In [ ]:
ggraph(se2@misc$cluster_ggraph, layout='tree', circular=T) + 
    geom_edge_diagonal2(aes(color=Lineage)) + theme_void() + 
    scale_edge_colour_brewer(palette=4)

In [ ]:
ggraph(se@misc$cluster_ggraph, layout='tree', circular=T) + 
    geom_edge_diagonal2(aes(color=Differentia)) + theme_void() + 
    scale_edge_colour_brewer()